# TabuLLM: Fraud Detection Walkthrough

This notebook demonstrates the core **TabuLLM** workflow on a fraud detection dataset:

1. **Traditional vectorization (TF-IDF)** — baseline approach and its limitations
2. **Embedding** text columns with `TextColumnTransformer`
3. **Dimensionality reduction** with `GMMFeatureExtractor`
4. **Cluster explanation** with `ClusterExplainer`
5. **Predictive pipeline** combining text and structured features

**Requirements:**
- `pip install tabullm[examples]`
- The embedding step uses `sentence-transformers/all-MiniLM-L6-v2` (free, runs locally).
- The `ClusterExplainer` step calls an LLM. Set `LLM_PROVIDER` below to `"openai"` or `"bedrock"`,
  and provide the corresponding credentials in a `.env` file or environment variables.
- All ClusterExplainer outputs are pre-computed in this notebook.
  Set `RERUN = True` below to re-execute the LLM calls.

In [11]:
# --- Configuration ---
RERUN = True          # Set to True to re-execute LLM calls
LLM_PROVIDER = "bedrock"  # "openai" or "bedrock"

In [9]:
# Load environment variables from .env file (API keys, AWS credentials, etc.)
from dotenv import load_dotenv
load_dotenv()  # looks for .env in the current directory or parents

True

## 1. Loading the Data

In [2]:
from tabullm import load_fraud

X, y, metadata = load_fraud()

print(f"Samples: {metadata['n_samples']}")
print(f"Fraud rate: {metadata['class_distribution']['fraud_rate']:.1%}")
print(f"\nColumns ({len(X.columns)}):")
print(f"  Text (7): {metadata['text_columns']}")
print(f"  Binary (3): {metadata['binary_columns']}")
print(f"  Categorical (5): {metadata['categorical_columns']}")

/Users/amahani/code/TabuLLM/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Samples: 17880
Fraud rate: 4.8%

Columns (15):
  Text (7): ['title', 'location', 'department', 'company_profile', 'description', 'requirements', 'benefits']
  Binary (3): ['telecommuting', 'has_company_logo', 'has_questions']
  Categorical (5): ['employment_type', 'required_experience', 'required_education', 'industry', 'function']


In [3]:
text_cols = metadata['text_columns']
X[text_cols].head(2)

,title,location,department,company_profile,description,requirements,benefits
0,Marketing Intern,"US, NY, New York",Marketing,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN
1,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...


## 2. Traditional Vectorization (TF-IDF)

scikit-learn's `TfidfVectorizer` produces sparse bag-of-words representations based on term
frequencies. While computationally efficient, these features capture lexical patterns (which
words appear) rather than semantic meaning (what the text means). Two job postings using
different vocabularies to describe the same role ("Software Engineer" versus "Code Developer")
would appear dissimilar despite semantic equivalence.

For datasets with multiple text columns, manual concatenation is required:

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Manual concatenation of all 7 text columns
texts = X[text_cols].fillna('').apply(
    lambda row: ' || '.join(row), axis=1
)

# Vectorize
vectorizer = TfidfVectorizer(max_features=384)
X_tfidf = vectorizer.fit_transform(texts).toarray()

print(f"TF-IDF features: {X_tfidf.shape}")

TF-IDF features: (17880, 384)


The following subsection demonstrates how `TextColumnTransformer` addresses both the semantic
limitation (by providing access to LLM embeddings that capture meaning) and the integration
challenge (through native multi-column handling within scikit-learn pipelines).

## 3. Embedding Text Columns with `TextColumnTransformer`

`TextColumnTransformer` wraps any LangChain-supported embedding model in a scikit-learn
`fit`/`transform` interface, handling multi-column concatenation natively:

In [5]:
from tabullm import TextColumnTransformer
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    # remove next two lines before committing the notebook
    cache_folder='/Users/amahani/Library/CloudStorage/OneDrive-ICEInc/Documents/code/github/TabuLLM/manuscript/jss-v2/Models',
    model_kwargs={'local_files_only': True},
)

text_transformer = TextColumnTransformer(model=embedding_model)
X_emb = text_transformer.fit_transform(X[text_cols])

print(f"LLM embeddings shape: {X_emb.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4075.17it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LLM embeddings shape: (17880, 384)


The result is a 17,880 x 384 dense matrix of semantic vectors. Compare the API to the TF-IDF
approach above: no manual concatenation, no `.fillna()`, no `.toarray()` — and the embeddings
capture semantic meaning rather than lexical overlap.

However, 384 dimensions are high-dimensional and opaque — motivating dimensionality reduction.

## 4. Dimensionality Reduction with `GMMFeatureExtractor`

`GMMFeatureExtractor` fits a Gaussian mixture model and returns per-cluster log-joint features,
reducing the embedding space from hundreds of dimensions to K interpretable features:

In [6]:
from tabullm import GMMFeatureExtractor

gmm = GMMFeatureExtractor(n_components=10, random_state=42)
X_features = gmm.fit_transform(X_emb)
cluster_labels = gmm.labels_

print(f"GMM log-joint features shape: {X_features.shape}")
print(f"Reduction: {X_emb.shape[1]}D -> {X_features.shape[1]}D")

GMM log-joint features shape: (17880, 10)
Reduction: 384D -> 10D


The 384-dimensional embedding is reduced to 10 log-joint features — one per GMM component.
Setting `include_log_density=True` appends the marginal log-density as an 11th feature,
which serves as an explicit outlier score.

### Cluster quality diagnostics

The `assignment_confidence_stats()` method returns per-observation diagnostics characterizing
how confidently each observation belongs to its assigned cluster:

In [7]:
obs_stats = gmm.assignment_confidence_stats(X_emb)
obs_stats.head()

,max_posterior,entropy,log_joint_margin,log_density
0,1.0,8.400815e-194,447.709797,1092.544703
1,1.0,8.907784e-259,597.319231,1134.216497
2,1.0,1.316880e-123,286.079318,1038.792000
3,1.0,3.885767e-154,356.377402,1052.550489
4,1.0,8.215763e-141,325.695061,1126.149399


The four columns are:
- **max_posterior**: highest posterior probability across clusters (1/K to 1)
- **entropy**: Shannon entropy of the posterior distribution (high = uncertain)
- **log_joint_margin**: gap between the top-two cluster log-joints (high = well-separated)
- **log_density**: marginal log-likelihood (low = outlier)

These diagnostics can be passed directly to `ClusterExplainer` for outcome-based analysis.

## 5. Cluster Explanation with `ClusterExplainer`

`ClusterExplainer` generates natural language descriptions of each cluster using LLMs,
with optional outcome-based statistical testing and narrative synthesis.

### Cost preview

Before making LLM calls, the `preview` flag estimates token usage and cost:

In [10]:
from tabullm import ClusterExplainer

# Initialize LLM based on provider choice
if LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    llm_cost = 0.15  # USD per 1M input tokens
elif LLM_PROVIDER == "bedrock":
    from langchain_aws import ChatBedrock
    llm = ChatBedrock(
        model_id='us.anthropic.claude-haiku-4-5-20251001-v1:0',
        region_name='us-east-1',
        model_kwargs={"max_tokens": 4096, "temperature": 0.0},
    )
    llm_cost = 1.00
else:
    raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER!r}")

print(f"Using LLM provider: {LLM_PROVIDER}")

explainer = ClusterExplainer(
    llm=llm,
    text_transformer=text_transformer,
    observations='job postings',
    text_fields='title, location, department, company profile, '
               'description, requirements, and benefits',
    llm_cost_per_1M_tokens=llm_cost,
)

explainer.explain(
    X, cluster_labels,
    max_members_per_cluster=100,
    random_state=42,
    preview=True
)

Using LLM provider: bedrock
Preview
───────
Tokens (estimated):   699,328
Cost (estimated):     $0.6993
Strategy:             two-stage (auto-selected)
Max members/cluster:  100



{'total_tokens': 699328, 'estimated_cost': 0.699328, 'strategy': 'two-stage'}

### Full explanation

The full call generates cluster descriptions (blind to the outcome), runs one-versus-rest
statistical tests, analyzes per-observation diagnostics, and produces a narrative synthesis:

In [ ]:
if RERUN:
    stat_labels = {
        'log_joint_margin': 'avg. margin between the top-2 cluster log-joints',
        'log_density': 'avg. log mixture density (typicality)',
    }

    result_df, global_results, stat_assoc_df, synthesis = explainer.explain(
        X, cluster_labels,
        y=y,
        y_label="fraudulent job posting (1=fraud, 0=legitimate)",
        observation_stats=obs_stats[['log_joint_margin', 'log_density']],
        stat_labels=stat_labels,
        synthesize=True,
        max_members_per_cluster=100,
        random_state=42,
    )

    # Save results for reproducibility
    import json
    results = {
        'result_df': result_df.to_dict(orient='records'),
        'global_results': global_results,
        'stat_assoc_df': stat_assoc_df.to_dict(orient='records'),
        'synthesis': synthesis,
    }
    with open('01_cluster_explanations.json', 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print("Results saved to 01_cluster_explanations.json")
else:
    import json
    import pandas as pd
    with open('01_cluster_explanations.json') as f:
        results = json.load(f)
    result_df = pd.DataFrame(results['result_df'])
    global_results = results['global_results']
    stat_assoc_df = pd.DataFrame(results['stat_assoc_df'])
    synthesis = results['synthesis']
    print("Loaded pre-computed results from 01_cluster_explanations.json")

### Cluster descriptions and fraud associations

In [ ]:
display_cols = ['cluster', 'short_label', 'size', 'OR', 'P-value']
available_cols = [c for c in display_cols if c in result_df.columns]
result_df[available_cols]

### Per-observation diagnostic associations

The `stat_assoc_df` shows how the GMM diagnostics (passed via `observation_stats`)
relate to the outcome:

In [ ]:
stat_assoc_df

### Synthesis narrative

The `synthesis` output connects cluster descriptions and statistical findings
into a coherent narrative:

In [ ]:
from IPython.display import Markdown
Markdown(synthesis)

## 6. Predictive Pipeline

The same TabuLLM components compose directly into a scikit-learn `Pipeline` that
combines text and structured features for prediction. We first establish a non-text
baseline, then add text features via TabuLLM.

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Column groups
binary_cols = metadata['binary_columns']
low_card_cols = ['employment_type', 'required_experience', 'required_education']
med_card_cols = ['industry', 'function']

### Baseline: non-text features only

In [17]:
baseline_pipeline = Pipeline([
    ('features', ColumnTransformer([
        ('binary', 'passthrough', binary_cols),
        ('low_card', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
        ]), low_card_cols),
        ('med_card', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('target', TargetEncoder(cv=5, smooth='auto', target_type='binary',
                                     random_state=42))
        ]), med_card_cols),
    ])),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

baseline_pipeline.fit(X_train, y_train)
baseline_auc = roc_auc_score(
    y_test, baseline_pipeline.predict_proba(X_test)[:, 1]
)
print(f"Baseline (non-text features only) Test AUC: {baseline_auc:.4f}")

Baseline (non-text features only) Test AUC: 0.9157


### Full pipeline: text + structured features

In [18]:
from tabullm import TextColumnTransformer, GMMFeatureExtractor
from sklearn.preprocessing import TargetEncoder

full_pipeline = Pipeline([
    ('features', ColumnTransformer([
        ('text', Pipeline([
            ('embed', TextColumnTransformer(model=embedding_model)),
            ('reduce', GMMFeatureExtractor(n_components=10,
                                           include_log_density=True,
                                           random_state=42))
        ]), text_cols),
        ('binary', 'passthrough', binary_cols),
        ('low_card', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
        ]), low_card_cols),
        ('med_card', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('target', TargetEncoder(cv=5, smooth='auto', target_type='binary',
                                     random_state=42))
        ]), med_card_cols),
    ])),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

full_pipeline.fit(X_train, y_train)
full_auc = roc_auc_score(
    y_test, full_pipeline.predict_proba(X_test)[:, 1]
)
print(f"Full pipeline (text + structured) Test AUC: {full_auc:.4f}")
print(f"Improvement over baseline: +{full_auc - baseline_auc:.4f}")

Full pipeline (text + structured) Test AUC: 0.9751
Improvement over baseline: +0.0594


## Next Steps

This notebook demonstrated the core TabuLLM workflow: embedding, dimensionality reduction,
cluster explanation, and predictive pipeline integration.

For advanced pipeline patterns — including column selection analysis and stacking ensembles
that push AUC further — see `02_advanced_pipelines.ipynb`.